In [1]:
## The following code ensures that all functions and init files are reloaded before executions.
%load_ext autoreload
%autoreload 2

In [2]:
import dask
dask.config.set({'dataframe.query-planning': False})
import spatialdata

c:\Users\ge37voy\AppData\Local\miniconda3\envs\sd313\Lib\site-packages\dask\dataframe\__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
c:\Users\ge37voy\AppData\Local\miniconda3\envs\sd313\Lib\site-packages\xarray_schema\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
c:\Users\ge37voy\AppData\Local\miniconda3\envs\sd313\Lib\site-packages\spatialdata\_core\query\relational_query.py:530: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap 

In [3]:
from insitupy.io import read_visium
from spatialdata_io import visium
from insitupy.datasets import xenium_human_breast_cancer, visium_human_breast_cancer

c:\Users\ge37voy\AppData\Local\miniconda3\envs\sd313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from pathlib import Path
from insitupy import CACHE, InSituData

In [5]:
# prepare paths
data_dir = Path(CACHE / "out/demo_insitupy_project") # directory of xenium data
xenium = InSituData.read(data_dir)
xenium.load_all()

In [6]:
visium = visium_human_breast_cancer()

INFO: Reading Visium data with spatialdata-io...


H5 file exists. Checking md5sum...
The h5 file md5sum matches. Download is skipped. To force download set `overwrite=True`.
Spatial directory exists. Download is skipped. To force download set `overwrite=True`.
Visium data structure is ready at C:\Users\ge37voy\.cache\InSituPy\demo_datasets\visium_hbreastcancer\CytAssist_FFPE_Human_Breast_Cancer
Dataset contains:
- filtered_feature_bc_matrix.h5
- spatial/ directory


c:\Users\ge37voy\AppData\Local\miniconda3\envs\sd313\Lib\site-packages\anndata\_core\anndata.py:1798: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
c:\Users\ge37voy\AppData\Local\miniconda3\envs\sd313\Lib\site-packages\anndata\_core\anndata.py:1798: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
c:\Users\ge37voy\AppData\Local\miniconda3\envs\sd313\Lib\site-packages\spatialdata\models\models.py:1183: UserWarning: Converting `region_key: region` to categorical dtype.
  convert_region_column_to_categorical(adata)
INFO: Using 'visium' coordinate system for pixel size extraction.


Adding images...


INFO: Converting 4992 Point geometries with radius to circular polygons using buffer.


## Add fullres image to Visium dataset

The fullres image is the one that needs later to be aligned with Xenium data.

In [7]:
from insitupy.images.io import read_image

In [8]:
fullres_path = r"C:\Users\ge37voy\.cache\InSituPy\demo_datasets\visium_hbreastcancer\CytAssist_FFPE_Human_Breast_Cancer\fullres_image_file.ome.tif"
fullres, ome_meta, axes, pixel_size = read_image(fullres_path)

In [9]:
fullres_img = fullres[0]

In [10]:
visium.images.add_image(
    image=fullres_img,
    axes=axes,
    pixel_size=pixel_size,
    name="fullres")

In [11]:
visium.images['fullres']

dask.array<from-zarr, shape=(21571, 19505, 3), dtype=uint8, chunksize=(1024, 1024, 3), chunktype=numpy.ndarray>

In [12]:
visium_out = CACHE / "out/features_test_fullres"

In [ ]:

visium.saveas(visium_out, overwrite=True)

In [13]:
from insitupy import InSituData

In [14]:
# load data
visium = InSituData.read(visium_out)
visium.load_all()

In [15]:
xenium_new = xenium.copy()

In [20]:
M_file = r"C:\Users\ge37voy\.cache\InSituPy\demo_datasets\Visium_HE_alignment_files\matrix.csv"

xenium_new.align_features(
    other=visium,
    transformation_matrix=M_file,
    source_image_name="fullres",
    reference_image_name="HE",
    transfer_images=True,
    verbose=True
)

Transforming and aligning features...
Converted transformation matrix from pixel coordinates (reference: 0.2125 µm/pixel) to physical coordinates.
Applying transformation (in physical coordinates): a=0.0560531105468679, b=1.0044753763751142, d=-1.0044753763751142, e=0.0560531105468679, xoff=-2616.775799489288, yoff=8535.07889349505
Transformed 4992 features.
Features aligned and added to InSituData object.
Transforming and aligning images...
Converted transformation matrix from pixel coordinates (reference: 0.2125 µm/pixel) to physical coordinates.
Applying transformation matrix (in physical coordinates):
[[ 5.60531105e-02  1.00447538e+00 -2.61677580e+03]
 [-1.00447538e+00  5.60531105e-02  8.53507889e+03]]
Transforming image 'fullres' with shape (21571, 19505, 3) -> output size (19505, 21571)
Transformed image 'fullres'
Transforming image 'hires' with shape (2000, 1809, 3) -> output size (1809, 2000)
Transformed image 'hires'
Transforming image 'lowres' with shape (600, 543, 3) -> outp

In [22]:
aligned_out = CACHE / "out/xenium_visium_aligned"

In [24]:
xenium_new.saveas(aligned_out, overwrite=True)

Saving data to C:\Users\ge37voy\.cache\InSituPy\out\xenium_visium_aligned


c:\Users\ge37voy\AppData\Local\miniconda3\envs\sd313\Lib\site-packages\zarr\core\dtype\npy\string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=6, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
2025-12-08 15:02:51 | [INFO] Created 28 records
2025-12-08 15:02:51 | [INFO] Created 5 records
2025-12-08 15:02:51 | [INFO] Created 7 records
2025-12-08 15:02:51 | [INFO] Created 18 records
2025-12-08 15:02:51 | [INFO] Created 18 records
2025-12-08 15:02:51 | [INFO] Created 6 records
2025-12-08 15:02:51 | [INFO] Created 3 records
2025-12-08 15

Saved.


In [25]:
xenium_feat = InSituData.read(aligned_out)
xenium_feat.load_all()

In [26]:
xenium_feat.show()

2025-12-08 15:02:58 | [WARNING] QWindowsWindow::setGeometry: Unable to set geometry 1086x655+2985-1161 (frame: 1102x694+2977-1192) on QWidgetWindow/"_QtMainWindowClassWindow" on "\\.\DISPLAY10". Resulting geometry: 723x500+2982-1175 (frame: 739x539+2974-1206) margins: 8, 31, 8, 8 minimum size: 385x500 MINMAXINFO maxSize=0,0 maxpos=0,0 mintrack=401,539 maxtrack=0,0)
